In [5]:
%pip install datasets transformers torch ipywidgets


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
# We got script errors for unsuported multi_eurlex.py as the datasets library used in the reference article has been moved away from custom Python scripts for security.
# We use LexGLUE benchmark versions which are pre-formatted for multi-label classification.
# We set a HF_TOKEN to enable higher rate limits and faster downloads.

In [7]:
%pip install python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
from dotenv import load_dotenv
import huggingface_hub

load_dotenv()

# Extract the token value
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    huggingface_hub.login(token=hf_token)
    print("Login status: Success - Authenticated securely")
else:
    print("Security Alert: No token found. Please check your local .env file")

HfHubHTTPError: Invalid user token. The token from HF_TOKEN environment variable is invalid. Note that HF_TOKEN takes precedence over `hf auth login`.

In [ ]:
from datasets import load_dataset
import os

# Load EUR-LEX (Part of LexGLUE benchmark). This uses the stable Parquet format which bypasses multi_eurlex.py errors
eurlex_ds = load_dataset("coastalcph/lex_glue", "eurlex")

print(f"EUR-LEX Train Documents: {len(eurlex_ds['train'])}")
print(eurlex_ds)

EUR-LEX Train Documents: 55000
DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 55000
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 5000
    })
})


In [ ]:
import requests

# The official Zenodo URL for the 69-label version
url = 'https://zenodo.org/records/6355465/files/uk-lex69.jsonl?download=1'
filename = 'uk-lex69.jsonl'

print(f"Downloading {filename}")
response = requests.get(url, stream=True)

uk_file = 'uk-lex69.jsonl'


uklex_ds = load_dataset('json', data_files=uk_file)
print(f"Total Documents: {len(uklex_ds['train'])}")

print(uklex_ds)

Total Documents: 36500
DatasetDict({
    train: Dataset({
        features: ['id', 'year', 'labels', 'title', 'body', 'data_type'],
        num_rows: 36500
    })
})


In [ ]:
# First document
if os.path.exists(uk_file):
    print("\nFirst Doc Title:", uklex_ds['train'][0].get('title', 'No title found'))
else:
    print("File still not found in the directory.")


First Doc Title: The Medicines (Products Other Than Veterinary Drugs) (General Sale List) Amendment Order 1995


To analyse concept drift we should split the data temporally.
Periods are splitted based on our reference paper Chalkidis(2022)
Dataset:

    EUR-LEX:
    Training 1958 – 2010
    Validation 2010 – 2012
    Testing 2012 – 2016

    UK-LEX:
    Training 1975 – 2002
    Validation 2003 – 2007
    Testing 2008 – 2018

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# convertion to DataFrame
df_uk = uklex_ds['train'].to_pandas()

# Modify 'year' column to numeric (int) 
# errors='coerce' turns any weird non-year text into "NaN" (Not a Number)
df_uk['year'] = pd.to_numeric(df_uk['year'], errors='coerce')


# Apply Chronological Splits (Chalkidis et al. 2022)
uk_train = df_uk[df_uk['year'] <= 2002]
uk_val   = df_uk[(df_uk['year'] > 2002) & (df_uk['year'] <= 2007)]
uk_test  = df_uk[df_uk['year'] >= 2008]

uklex_split = DatasetDict({
    'train': Dataset.from_pandas(uk_train.reset_index(drop=True)),
    'validation': Dataset.from_pandas(uk_val.reset_index(drop=True)),
    'test': Dataset.from_pandas(uk_test.reset_index(drop=True))
})

print("\nUK-LEX CHRONOLOGICAL SPLITS")
print(f"Train (1975-2002): {len(uklex_split['train'])} docs")
print(f"Val   (2003-2007): {len(uklex_split['validation'])} docs")
print(f"Test  (2008-2018): {len(uklex_split['test'])} docs")


UK-LEX CHRONOLOGICAL SPLITS
Train (1975-2002): 21391 docs
Val   (2003-2007): 6128 docs
Test  (2008-2018): 8981 docs


In [ ]:
print("EUR-LEX CHRONOLOGICAL SPLITS")
for split_name in eurlex_ds.keys():
    print(f"{split_name.capitalize()}: {len(eurlex_ds[split_name])} documents")

# Check "year" column
sample = eurlex_ds['train'][0]
if 'year' in sample:
    print(f"\nYear metadata found")
else:
    print("\nThe 'year' column is hidden in this benchmark version.")
    print("However, the splits are chronologically ordered as follows:")
    print(" - Train: ~1958 to 2010")
    print(" - Validation: 2010 to 2012")
    print(" - Test: 2012 to 2016 (The 'Digital Turn' period)")

EUR-LEX CHRONOLOGICAL SPLITS
Train: 55000 documents
Test: 5000 documents
Validation: 5000 documents

The 'year' column is hidden in this benchmark version.
However, the splits are chronologically ordered as follows:
 - Train: ~1958 to 2010
 - Validation: 2010 to 2012
 - Test: 2012 to 2016 (The 'Digital Turn' period)


In [ ]:
# DatasetDict objects with 'train', 'validation' and 'test'
eu_data = eurlex_ds
uk_data = uklex_split

In [ ]:
# Extract the names from the EuroVoc subset
eu_label_names = eu_data['train'].features['labels'].feature.names

print(f"EUR-LEX TOP 100 LABELS")
for i in range(0, 100, 4):
    chunk = eu_label_names[i:i+4]
    formatted_chunk = " | ".join([f"ID {i+j}: {name}" for j, name in enumerate(chunk)])
    print(formatted_chunk)

EUR-LEX TOP 100 LABELS
ID 0: 100163 | ID 1: 100168 | ID 2: 100169 | ID 3: 100170
ID 4: 100171 | ID 5: 100172 | ID 6: 100173 | ID 7: 100174
ID 8: 100175 | ID 9: 100176 | ID 10: 100177 | ID 11: 100179
ID 12: 100180 | ID 13: 100183 | ID 14: 100184 | ID 15: 100185
ID 16: 100186 | ID 17: 100187 | ID 18: 100189 | ID 19: 100190
ID 20: 100191 | ID 21: 100192 | ID 22: 100193 | ID 23: 100194
ID 24: 100195 | ID 25: 100196 | ID 26: 100197 | ID 27: 100198
ID 28: 100199 | ID 29: 100200 | ID 30: 100201 | ID 31: 100202
ID 32: 100204 | ID 33: 100205 | ID 34: 100206 | ID 35: 100207
ID 36: 100212 | ID 37: 100214 | ID 38: 100215 | ID 39: 100220
ID 40: 100221 | ID 41: 100222 | ID 42: 100223 | ID 43: 100224
ID 44: 100226 | ID 45: 100227 | ID 46: 100229 | ID 47: 100230
ID 48: 100231 | ID 49: 100232 | ID 50: 100233 | ID 51: 100234
ID 52: 100235 | ID 53: 100237 | ID 54: 100238 | ID 55: 100239
ID 56: 100240 | ID 57: 100241 | ID 58: 100242 | ID 59: 100243
ID 60: 100244 | ID 61: 100245 | ID 62: 100246 | ID 63: 10

ID,EuroVoc Code (Chalkidis)
65,100249,Protection of Privacy
66,100250,Personal Data
6,100173,Rights of the Individual
82,100268,Information Technology
36,100212,Child

In [ ]:
from collections import Counter

# All labels in the Test Set
all_test_labels = []
for doc in uk_data['test']:
    all_test_labels.extend(doc['labels'])

label_counts = Counter(all_test_labels)

print("10 Most Frequent Label IDs in UK-LEX Test Set")
for label_id, count in label_counts.most_common(10):
    print(f"ID {label_id}: appears in {count} documents")

10 Most Frequent Label IDs in UK-LEX Test Set
ID TAXATION: appears in 1325 documents
ID HEALTH CARE: appears in 1113 documents
ID EDUCATION: appears in 739 documents
ID LOCAL GOVERNMENT: appears in 711 documents
ID SOCIAL SECURITY: appears in 657 documents
ID NHS: appears in 584 documents
ID PENSIONS: appears in 570 documents
ID CHILDREN: appears in 494 documents
ID PLANNING: appears in 381 documents
ID TRAFFIC: appears in 367 documents


Following the reference paper, models struggle with "rare" labels during temporal shifts.